# Genetic Algorithm Lab: OneMax

In this exercise, you will complete a genetic algorithm for the OneMax problem. The notebook provides the experiment setup and the main optimization loop. Your task is to fill in the parameter values and implement three operators:

1. uniform crossover;
2. standard bit mutation; and
3. fitness-proportional mating selection.

Search budget: **10,000 objective-function evaluations per run**.

## 1. Setup

This exercise uses the `ioh` package for its benchmark problem implementation. If `ioh` is not installed in the current notebook environment, uncomment and run the installation command below once.

Documentation for the built-in pseudo-Boolean problems is available on the [IOHprofiler PBO problem page](https://iohprofiler.github.io/IOHproblem/PBO).

In [1]:
# Uncomment the following line only if `ioh` is not installed.
# %pip install ioh

In [1]:
import time
from typing import List, Optional, Tuple

import numpy as np
from ioh import ProblemClass, get_problem

## 2. The OneMax problem

We use the CountingOnes/OneMax problem introduced in class:

$$
f_{\mathrm{OM}}(\mathbf{x}) = \sum_{i=1}^{d} x_i,
\qquad \mathbf{x} \in \{0,1\}^{d}.
$$

The goal is to maximize the number of one-bits. Therefore, the optimal fitness value is $d$.

In [2]:
dimension = 50

# Function ID 1 in the PBO suite is OneMax.
om = get_problem(
    1,
    dimension=dimension,
    instance=1,
    problem_class=ProblemClass.PBO,
)

target_fitness = float(dimension)

The IOH problem object is callable. For example, evaluating the all-ones bit string should return the optimal fitness value, 50.

In [3]:
example_fitness = om(np.ones(dimension, dtype=int))
om.reset()
example_fitness

50.0

## 3. Choose the parameter values

Replace each `...` with a suitable value. The mutation rate is the probability of flipping each bit. The crossover probability is the probability of applying crossover to a selected pair of parents.

In [34]:
# TODO: Choose the genetic algorithm parameters.
pop_size = 20
mutation_rate = 1.0 / dimension
crossover_probability = 0.8

## 4. Implement the variation and selection operators

Follow these conventions so that the operators work correctly with the provided genetic algorithm:

- An individual is a one-dimensional NumPy array containing zeros and ones.
- `crossover` must return **two new arrays** and must not modify its input parents. For uniform crossover, decide independently at each position which parent contributes the bit.
- `mutation` must return a new mutated array and must not modify its input individual. Flip every bit independently with probability `rate`.
- `mating_selection` must return `len(parents)` independently copied individuals. First shift the fitness values by their minimum, then sample with replacement using probabilities proportional to the shifted values. If all shifted values are zero, use uniform probabilities.

In [14]:
def crossover(
    parent_1: np.ndarray,
    parent_2: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Create two offspring using uniform crossover."""
    # `crossover_probability` is handled in the main GA loop. Inside
    # uniform crossover, each bit position is swapped with rate 0.5.
    child_1, child_2 = np.zeros(dimension), np.zeros(dimension)
    for i_1, bit_1 in enumerate(child_1):
        pbit1, pbit2 = parent_1[i_1], parent_2[i_1]
        chosen_bit = np.random.choice([pbit1, pbit2])
        child_1[i_1] = chosen_bit
    
    for i_2, bit_2 in enumerate(child_2):
        pbit1, pbit2 = parent_1[i_2], parent_2[i_2]
        chosen_bit = np.random.choice([pbit1, pbit2])
        child_2[i_2] = chosen_bit
    
    return child_1, child_2
    

In [16]:
parent_1 = np.zeros(dimension)
parent_2 = np.ones(dimension)

child_1, child_2 = crossover(parent_1, parent_2)

print(child_1)
print(sum(child_1))
print(child_2)
print(sum(child_2))

[1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 1.
 1. 0. 0. 1. 0. 0. 1. 0. 1. 1. 0. 1. 0. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1. 0.
 1. 1.]
32.0
[0. 0. 1. 1. 1. 0. 1. 1. 0. 1. 1. 0. 1. 1. 1. 0. 0. 0. 1. 1. 1. 1. 1. 0.
 0. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 1. 1. 0. 0.
 1. 1.]
28.0


In [18]:
def mutation(individual: np.ndarray, rate: float) -> np.ndarray:
    """Apply standard bit mutation and return a new individual."""
    new_individual = individual.copy()
    decisions = [True, False]
    for i, bit in enumerate(new_individual):
        p = [rate, 1 - rate]
        flip = np.random.choice(decisions, p=p)
        if flip:
            if new_individual[i] == 0:
                new_individual[i] = 1
            else:
                new_individual[i] = 0
    
    return new_individual
        

In [28]:
individual = np.ones(dimension)
new_individual = mutation(individual, rate=0.3)

print(sum(individual))
print(sum(new_individual))

50.0
35.0


In [30]:
def mating_selection(
    parents: List[np.ndarray],
    parent_fitness: List[float],
) -> List[np.ndarray]:
    """Select a mating pool using fitness-proportional selection."""
    # Before roulette-wheel selection, shift every fitness value by the
    # minimum fitness: shifted_fitness = fitness - min(parent_fitness).
    # This keeps the ranking unchanged and makes all selection weights
    # non-negative. If all shifted values are zero, select uniformly.
    min_f = min(parent_fitness)
    shifted_fitness = [f - min_f for f in parent_fitness]
    total_f = sum(shifted_fitness)
    
    n_parents = len(parents)
    if total_f == 0:
        p = [1.0 / n_parents] * n_parents
    else:
        p = [f / total_f for f in shifted_fitness]
    
    selected_indices = np.random.choice(n_parents, size=n_parents, p=p)
    return [parents[i].copy() for i in selected_indices]

## 5. Genetic algorithm

The main loop is provided below. You do not need to modify it. It initializes the population, calls your three operators, evaluates the offspring, keeps track of the best solution, and never exceeds the evaluation budget.

In [31]:
def genetic_algorithm(
    func,
    target: float,
    budget: int = 10_000,
) -> Tuple[float, np.ndarray]:
    """Run one independent genetic algorithm trial."""
    if not isinstance(pop_size, int) or isinstance(pop_size, bool) or pop_size < 2:
        raise ValueError("pop_size must be an integer of at least 2.")
    if budget < pop_size:
        raise ValueError("budget must be at least as large as pop_size.")
    if not 0.0 <= mutation_rate <= 1.0:
        raise ValueError("mutation_rate must lie in [0, 1].")
    if not 0.0 <= crossover_probability <= 1.0:
        raise ValueError("crossover_probability must lie in [0, 1].")

    # Start every run with a clean IOH problem state.
    func.reset()
    n_variables = func.meta_data.n_variables
    parents = [
        np.random.randint(0, 2, size=n_variables, dtype=int)
        for _ in range(pop_size)
    ]

    # Evaluate the initial population and record its best individual.
    parent_fitness: List[float] = []
    evaluations = 0
    best_fitness = -np.inf
    best_individual: Optional[np.ndarray] = None

    for individual in parents:
        fitness = float(func(individual))
        parent_fitness.append(fitness)
        evaluations += 1

        if fitness > best_fitness:
            best_fitness = fitness
            best_individual = individual.copy()

    while best_fitness < target and evaluations < budget:
        mating_pool = mating_selection(parents, parent_fitness)
        if len(mating_pool) != len(parents):
            raise ValueError("mating_selection must preserve the population size.")

        # The last generation may be smaller when little budget remains.
        n_offspring = min(pop_size, budget - evaluations)
        offspring: List[np.ndarray] = []
        pair_index = 0

        while len(offspring) < n_offspring:
            first = (2 * pair_index) % pop_size
            second = (first + 1) % pop_size
            parent_1 = np.asarray(mating_pool[first], dtype=int).copy()
            parent_2 = np.asarray(mating_pool[second], dtype=int).copy()

            if np.random.random() < crossover_probability:
                child_1, child_2 = crossover(parent_1, parent_2)
            else:
                child_1, child_2 = parent_1.copy(), parent_2.copy()

            child_1 = mutation(np.asarray(child_1, dtype=int), mutation_rate)
            child_2 = mutation(np.asarray(child_2, dtype=int), mutation_rate)
            offspring.append(np.asarray(child_1, dtype=int))
            if len(offspring) < n_offspring:
                offspring.append(np.asarray(child_2, dtype=int))

            pair_index += 1

        offspring_fitness: List[float] = []
        for individual in offspring:
            fitness = float(func(individual))
            offspring_fitness.append(fitness)
            evaluations += 1

            if fitness > best_fitness:
                best_fitness = fitness
                best_individual = individual.copy()

        # Generational replacement. A shortened final generation is safe
        # because the evaluation budget is exhausted immediately afterward.
        parents = offspring
        parent_fitness = offspring_fitness

    if best_individual is None:
        raise RuntimeError("No individual was evaluated.")

    func.reset()
    return best_fitness, best_individual

## 6. Run the experiment

After completing all `TODO` sections, run the following cells. Setting `base_seed` to an integer makes the experiment reproducible. Set it to `None` to obtain different random runs each time.

In [32]:
def run_experiment(
    n_runs: int = 20,
    budget: int = 10_000,
    base_seed: Optional[int] = 42,
) -> List[float]:
    """Run the genetic algorithm repeatedly and return the best fitness values."""
    best_values: List[float] = []

    for run in range(n_runs):
        if base_seed is not None:
            np.random.seed(base_seed + run)

        best_fitness, _ = genetic_algorithm(
            om,
            target=target_fitness,
            budget=budget,
        )
        best_values.append(best_fitness)
        print(f"Run {run + 1:02d}: best fitness = {best_fitness:g}")

    return best_values

In [37]:
start_time = time.perf_counter()
results = run_experiment()
elapsed_time = time.perf_counter() - start_time

print(f"Best fitness over all runs: {max(results):g}")
print(f"Total running time: {elapsed_time:.2f} seconds")

Run 01: best fitness = 50


KeyboardInterrupt: 